# Dataset Creation

The goal of this notebook is to create a curated training dataset of protein sequences and secondary structure labels.

## Download Protein Structures

A subset of representative training structures was curating using the [PISCES](https://dunbrack.fccc.edu/pisces/PISCES_OptionPage.php) tool from the Dunbrack lab at FCCC. PDB IDs and Chain IDs from the `selected_protein.txt` are then downloaded from the Protein Data Bank (PDB) using their web API. 

In [ ]:
import urllib.request, pathlib, time
from tqdm.auto import tqdm

In [5]:
out = pathlib.Path("Structures")
out.mkdir(exist_ok=True)

def parse_pisces(path):
    ids = []
    with open(path) as f:
        next(f)
        for line in f:
            tok = line.split()[0]
            ids.append((tok[:4].lower(), tok[4:])) # (pdb_id, chain)
    return ids

entries = parse_pisces("selected_chains.txt")

for pdb_id in tqdm(sorted({p for p, _ in entries}), desc="Structure File Download"):
    dest = out / f"{pdb_id}.cif"
    if dest.exists():
        continue
    try:
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.cif", dest)
    except Exception as e:
        print(f"failed {pdb_id}: {e}")
    time.sleep(0.1)

Structure File Download:   0%|          | 0/8445 [00:00<?, ?it/s]

## Assignment of Secondary Structure

Each structure file (`.cif`) is passed through the standard secondary structure assignment tool [DSSP](https://github.com/PDB-REDO/dssp). This tool is built from binary locally using CMake and is not included in the repository.

Each residue is assigned a secondary structure based on electrostatic interactinos, backbone hydrogen-bonding patterns, and torsion angles. It provides 9 structural assignments at each residue position:
- H: $\alpha$-Helix
- G: $3_{10}$-Helix
- I: $\pi$-Helix
- E: $\beta$-Sheet
- B: $\beta$-Bridge
- T: Hydrogen-Bonded Turn
- S: High Curvature
- P: Polyproline II Helix
- C: Coil/Loop

I redefine this to just 3 labels:
- H,G,I &rarr; H: Helix
- E,B &rarr; E: Sheet
- T,S,P,C &rarr; C: Coil

I finally create a dataframe containing the PDB ID, Chain ID, Amino Acid Sequence, and Secondary Structure Label saved to `selected_proteins.csv`.

In [6]:
from collections import defaultdict
from Bio.PDB import MMCIFParser, DSSP
import pandas as pd
import os, sys, shutil
import warnings

# mkdssp and its data folder
MKDSSP = os.path.join(sys.prefix, "bin", "mkdssp")
if not os.path.exists(MKDSSP):
    MKDSSP = shutil.which("mkdssp")
assert MKDSSP, "mkdssp not found; run `conda install -c conda-forge dssp` in this environment"

DATA_DIR = os.path.join(sys.prefix, "share", "libcifpp")
assert os.path.exists(os.path.join(DATA_DIR, "mmcif_pdbx.dic")), \
    f"mmcif_pdbx.dic not found in {DATA_DIR}"
os.environ["LIBCIFPP_DATA_DIR"] = DATA_DIR

print("mkdssp:", MKDSSP)
print("data dir:", DATA_DIR)

three_state = {
    **dict.fromkeys("HGI", "H"),
    **dict.fromkeys("EB", "E"),
}  # everything else -> "C"

# Add chain_id to pdb_id for DSSP label
wanted = defaultdict(set)
for pdb_id, chain_id in entries:
    wanted[pdb_id].add(chain_id)

warnings.filterwarnings(
    "ignore",
    message=r".*version in dictionary mmcif_pdbx\.dic is lower than requested.*",
    category=UserWarning,
)

rows = []
for pdb_id, chains in tqdm(wanted.items(), desc="DSSP Processing"):
    path = f"Structures/{pdb_id}.cif"
    try:
        s = MMCIFParser(QUIET=True).get_structure(pdb_id, path)
        dssp = DSSP(s[0], path, dssp=MKDSSP, file_type="MMCIF")
    except Exception as e:
        tqdm.write(f"skipping {pdb_id}: {e}")
        continue

    seq, ss = defaultdict(list), defaultdict(list)
    for (ch, _), v in dssp.property_dict.items():
        if ch in chains:
            seq[ch].append(v[1])
            ss[ch].append(three_state.get(v[2], "C"))

    for ch in chains:
        if seq[ch]:
            rows.append({
                "pdb_id": pdb_id,
                "chain": ch,
                "sequence": "".join(seq[ch]),
                "structure": "".join(ss[ch]),
            })

df = pd.DataFrame(rows)
df.to_csv("selected_proteins.csv", index=False)

mkdssp: /opt/miniconda3/envs/protein_dataset_env/bin/mkdssp
data dir: /opt/miniconda3/envs/protein_dataset_env/share/libcifpp


DSSP Processing:   0%|          | 0/8445 [00:00<?, ?it/s]

skipping 1gci: [Errno 60] Operation timed out


/opt/miniconda3/envs/protein_dataset_env/lib/python3.14/site-packages/Bio/PDB/DSSP.py:199: UserWarning: The data in this file won't fit in the old DSSP format, please use the mmCIF format instead.

  warnings.warn(err)


skipping 7af2: DSSP failed to produce an output
skipping 1g6x: [Errno 60] Operation timed out


KeyboardInterrupt: 

In [ ]:
print(len(df), "chains written")

8586 chains written
